# Gobernanza de Metadatos Semánticos y Knowledge Graphs (Aplicado)

## Objetivos
- Implementar gobierno de metadatos semánticos para discovery y reutilización.
- Construir un mini grafo de conocimiento (datasets, columnas, conceptos, dominios).
- Ejecutar consultas SPARQL para descubrir datasets relevantes por concepto/KPI.
- Validar contratos/propiedades mínimas de assets con verificaciones simples.
- Conectar con catálogo y data contracts para interoperabilidad.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Modelado Semántico Aplicado: Ontologías, RDF/OWL, SKOS y Knowledge Graphs](09_modelado_semantico_ontologias_kg.ipynb) | Siguiente: [Arquitecturas Cloud y Multi-Cloud (Aplicado)](11_arquitecturas_cloud_y_multi_cloud.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. Conceptos y Estándares
3. 2. Modelo de Grafo para Discovery/Reutilización
4. 3. Consultas SPARQL de Descubrimiento/Reutilización
5. 4. Validación Simple de Metadatos Requeridos
6. 5. Integración con Catálogo y Contratos
7. 6. Ejercicio

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Gobernanza de Metadatos Semánticos y Knowledge Graphs (Aplicado)"] --> C["Notebook siguiente"]
```


## 1. Conceptos y Estándares
- **Metadatos semánticos**: describen significado (no solo formato) usando vocabularios controlados.
- **Catálogo** (DataHub/Atlas/Collibra): sistema de registro de assets.
- **DCAT**: vocabulario W3C para describir datasets y distribución.
- **SKOS**: vocabulario para taxonomías (concepts, broader/narrower/related).
- **RDF/OWL**: modelo de grafo y lenguaje de ontologías.
- **SPARQL**: consultas sobre grafos RDF.

## 2. Modelo de Grafo para Discovery/Reutilización
Tipos de nodos y relaciones:
- `Dataset` → `hasTable` → `Table` → `hasColumn` → `Column`.
- `Column` → `mapsToConcept` → `Concept` (SKOS/OWL).
- `Dataset` → `belongsToDomain` → `Domain`.
- `Dataset` → `partOfProduct` → `DataProduct`.
- `KPI` → `usesConcept` → `Concept`.
Esto permite responder: ¿qué datasets soportan el KPI X?, ¿qué columnas contienen PII?, ¿qué alternativas existen para reutilizar?

In [ ]:
try:
    from rdflib import Graph, Namespace, RDF, RDFS, Literal
    from rdflib.namespace import SKOS, DCTERMS
    HAS_RDFLIB = True
except ModuleNotFoundError:
    HAS_RDFLIB = False

if HAS_RDFLIB:
    EX = Namespace('https://example.com/data/')
    g = Graph()
    g.bind('ex', EX)
    g.bind('skos', SKOS)
    g.bind('dct', DCTERMS)
    for cls in ['Dataset', 'Table', 'Column', 'Concept', 'Domain', 'DataProduct', 'KPI']:
        g.add((EX[cls], RDF.type, RDFS.Class))
    for concept, label in [(EX['concept/customer'], 'Customer'), (EX['concept/revenue'], 'Revenue')]:
        g.add((concept, RDF.type, SKOS.Concept))
        g.add((concept, SKOS.prefLabel, Literal(label)))
    ds_orders = EX['dataset/sales_orders_silver']
    g.add((ds_orders, RDF.type, EX.Dataset))
    g.add((ds_orders, DCTERMS.title, Literal('Sales Orders Silver')))
    print(g.serialize(format='turtle'))
else:
    catalogo_semantico = [
        {'dataset': 'sales_orders_silver', 'conceptos': ['Customer', 'Revenue']},
        {'dataset': 'marketing_customers_silver', 'conceptos': ['Customer', 'PII Email']}
    ]
    for item in catalogo_semantico:
        print(item)


## 3. Consultas SPARQL de Descubrimiento/Reutilización
1) Datasets que soportan el KPI ARPU (por conceptos usados)
2) Datasets con columnas marcadas como PII (ej: email)
3) Alternativas de reutilización por dominio

In [ ]:
if HAS_RDFLIB:
    q1 = '''
    PREFIX ex: <https://example.com/data/>
    PREFIX dct: <http://purl.org/dc/terms/>
    SELECT DISTINCT ?ds ?title
    WHERE { ?ds a ex:Dataset ; dct:title ?title . }
    '''
    for row in g.query(q1):
        print('Dataset disponible:', row.title)
else:
    print('KPI ARPU soportado por dataset: sales_orders_silver')
    print('Datasets con PII (email): marketing_customers_silver')
    print('Conteo de datasets por dominio: sales=1, marketing=1')


## 4. Validación Simple de Metadatos Requeridos
Validamos que cada `Dataset` tenga título y dominio asignado (simulación de SHACL mínima).

In [ ]:
# Validación ad-hoc (SHACL simplificado)
if HAS_RDFLIB:
    def validar_datasets(graph):
        problemas = []
        for ds in graph.subjects(RDF.type, EX.Dataset):
            title = list(graph.objects(ds, DCTERMS.title))
            domain = list(graph.objects(ds, EX.belongsToDomain))
            if not title or not domain:
                problemas.append((ds, 'Faltan metadatos: ' + ('title ' if not title else '') + ('domain' if not domain else '')))
        return problemas

    issues = validar_datasets(g)
    print('Problemas encontrados:' if issues else 'Todos los datasets cumplen metadatos mínimos')
    for issue in issues:
        print('-', issue)
else:
    print('Validación conceptual: cada dataset debe tener título, dominio y owner antes de publicarse.')


## 5. Integración con Catálogo y Contratos
- Mapear IDs del catálogo a IRIs del grafo (enlace estable).
- Enriquecer assets con `mapsToConcept` para mejorar búsqueda semántica.
- Publicar facetas de búsqueda por `Concept` y `Domain`.
- Exponer API: `GET /search?concept=customer` → datasets candidatos.

## 6. Ejercicio
1) Agrega un nuevo KPI `Churn` y enlázalo a conceptos `Customer` y `Status`.
2) Crea un dataset nuevo con columna `status` mapeada a `Status`.
3) Escribe una consulta para listar datasets que soportan el KPI Churn.
4) Extiende la validación para exigir `partOfProduct` cuando `belongsToDomain` sea `marketing`.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- DAMA International: https://www.dama.org/
- DataHub: https://datahubproject.io/
- W3C SPARQL: https://www.w3.org/TR/sparql11-query/
- Referencias del curso: ../04-recursos/referencias.md
